In [1]:
!pip install -q -U transformers accelerate peft sentencepiece bitsandbytes
!pip uninstall -y -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(UserSecretsClient().get_secret("HF_TOKEN"))

In [3]:
# Mix of clean conversational prompts and noisy, ASR-output-like prompts
# (lowercase-ish, no punctuation, a plausible misrecognition) to test
# robustness, since real input will come from imperfect ASR.
prompts_am = [
    "ሰላም እንዴት ነህ",
    "የአዲስ አበባ የአየር ሁኔታ ዛሬ እንዴት ነው",
    "አንድ ቀላል የኢትዮጵያ ምግብ አዘገጃጀት ንገረኝ",
    "ነገ ስብሰባ አለኝ አስታውሰኝ",
    "የኢትዮጵያ ዋና ከተማ ማናት",
    "ዛሬ ጠዋት ስለተፈጠረው ነገር ንገረኝ",   # intentionally vague, like noisy ASR
    "አዲስ አበባ ጎዳና ትራፊክ እንዴት ነው",
    "አንድ አጭር ቀልድ ንገረኝ",
]

print(f"{len(prompts_am)} prompts ready")

8 prompts ready


In [4]:
import time, csv, os
import torch
from threading import Thread
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

LOG_PATH = "/kaggle/working/llm_benchmark_log.csv"

def benchmark_llm(model_id, prompts, max_new_tokens=120, load_kwargs=None, model_label=None):
    label = model_label or model_id
    load_kwargs = load_kwargs or {}
    print(f"Loading {label} ...")
    t_load = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map="auto", **load_kwargs
    )
    print(f"Load time: {time.perf_counter() - t_load:.1f}s (one-time cost)")

    warm = tok("ሰላም", return_tensors="pt", return_dict=True).to(model.device)
    model.generate(**warm, max_new_tokens=8)

    rows = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)

        streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
        gen_kwargs = dict(**inputs, max_new_tokens=max_new_tokens, streamer=streamer)

        def _run(kwargs=gen_kwargs, streamer=streamer):
            try:
                model.generate(**kwargs)
            except Exception as e:
                print(f"  [generation error] {e}")
            finally:
                streamer.end()   # <-- guarantees the main thread never hangs, even on failure

        t0 = time.perf_counter()
        thread = Thread(target=_run)
        thread.start()

        first_token_time, pieces = None, []
        for piece in streamer:
            if first_token_time is None:
                first_token_time = time.perf_counter() - t0
            pieces.append(piece)
        thread.join(timeout=60)  # safety net: never wait more than 60s per prompt
        total = time.perf_counter() - t0
        text = "".join(pieces)
        n_tok = len(tok(text).input_ids) if text else 0

        rows.append({
            "model": label, "prompt": prompt,
            "ttft_sec": round(first_token_time or total, 3),
            "total_sec": round(total, 3),
            "tokens_generated": n_tok,
            "tokens_per_sec": round(n_tok / total, 2) if total > 0 and n_tok else 0,
            "response": text,
        })
        print(f"  {rows[-1]['total_sec']:.2f}s | {n_tok} tok -> {text[:60] or '[FAILED]'}")

    write_header = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        if write_header:
            w.writeheader()
        w.writerows(rows)

    del model
    torch.cuda.empty_cache()

    ok_rows = [r for r in rows if r["tokens_generated"] > 0]
    if ok_rows:
        ttfts = sorted(r["ttft_sec"] for r in ok_rows)
        print(f"\n{label} summary: median TTFT {ttfts[len(ttfts)//2]:.2f}s, "
              f"mean tok/s {sum(r['tokens_per_sec'] for r in ok_rows)/len(ok_rows):.1f}\n")
    else:
        print(f"\n{label}: every prompt failed, check the error messages above.\n")
    return rows

In [5]:
from peft import PeftModel

base_id = "google/gemma-3-4b-it"
adapter_id = "Reubencf/gemma-3-4b-it-amharic-lora"

tok = AutoTokenizer.from_pretrained(adapter_id)
base = AutoModelForCausalLM.from_pretrained(base_id, dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, adapter_id)

import time as _time
rows = []
for prompt in prompts_am:
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,          # <-- forces a proper dict with input_ids / attention_mask
    ).to(model.device)

    t0 = _time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=120)   # <-- unpack as kwargs, not positional
    total = _time.perf_counter() - t0

    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"{total:.2f}s -> {text[:80]}")
    rows.append({"model": "gemma-3-4b-amharic-lora", "prompt": prompt, "total_sec": round(total, 3), "response": text})

import pandas as pd
pd.DataFrame(rows).to_csv("/kaggle/working/gemma3_lora_results.csv", index=False)
del model, base
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/231 [00:00<?, ?B/s]

...

adapter_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  477MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.1.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_mode

6.43s -> እኔ ጥሩ ነኝ, አመነህ። ዛሬ እንዴት ነህ? ምን ማድረግ ትችላለህ?
16.83s -> በአሁኑ ጊዜ የ የአዲስ አበባ የአየር ሁኔታ፡

*   **तापпература:** 28° ሴልስየስ
*   **ደማቁማጭ:** ሰማይ 
17.00s -> አዎ፣ ለእናንተ ቀላል የኢትዮጵያ ምግብ አዘገጃጀት ዘዴ ልናነብ። አሁንaremos “ሚንቸ” ( መስጠት) የምንብብ አዘገጃጀት አድ
3.23s -> አዎ፣ ነገ ስብሰባ አለዎት ብዬ recuerdo። በምን ጊዜ እናት reunión ነው።
2.36s -> የኢትዮጵያ ዋና ከተማ አዲስ አበባ ነው።

15.84s -> ዛજે утроमा नेपालमा शक्तिशाली भूकम्प आएको छ। केन्द्रीय भूकम्पमालय विभागका अनुसार 
16.83s -> አዲስ አበባ ጎዳና ትራፊክ በአሁን ወቅት በጣም ጫና ነው። በተ特别是 የስልጠና ተማሪዎች እና የከተማ planificación dep
12.09s -> ለምን ፐርፖል ዛፍን ተከተለ?

因为他想看风从哪里吹来！ (因为他想看风从哪里吹来!)

ትርጉም: ምክንያቱም ಅವನು variaciónን எங


In [6]:
benchmark_llm("CohereLabs/tiny-aya-earth", prompts_am, model_label="tiny-aya-earth")

Loading tiny-aya-earth ...


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Load time: 52.5s (one-time cost)
  2.81s | 42 tok -> ሰላም! እንዴት ነህ? እኔ Aya ነኝ፣ Cohere የፈጠረችው AI ረዳት። እንዴት ልረዳህ እችላ
  7.61s | 121 tok -> የአዲስ አበባ የአየር ሁኔታ ዛሬ (በጁን 2024) በአጠቃላይ **በጣም ሙቅ እና ደረቅ** ነው።
  7.67s | 122 tok -> እርግጠኛ ነኝ! እዚህ ላይ አንድ ቀላል የኢትዮጵያ ምግብ አዘገጃጀት አለ፡ **ቲቦ (ቲቦ ስጋ)*
  7.64s | 121 tok -> እርግጠኛ ነኝ! እዚህ ላይ ለነገ ስብሰባዎ አስታውሰኝ ማስታወሻ አለ።

**"ነገ (ቀን X) በሰ
  4.90s | 75 tok -> የኢትዮጵያ ዋና ከተማ አዲስ አበባ ነው። አዲስ አበባ በኢትዮጵያ ፖለቲካ፣ ኢኮኖሚ፣ ባህል እና 
  7.70s | 120 tok -> ዛሬ ጠዋት ስለተፈጠረው ነገር ንገረኝ። ይህ ጥያቄ በአማርኛ "What happened today m
  7.59s | 121 tok -> አዲስ አበባ ጎዳና ትራፊክ በአጠቃላይ የተወሳሰበ እና የተለያዩ ችግሮች ያሉት ነው። በከተማዋ ው
  7.61s | 121 tok -> አንድ ሰው ወደ ሱቅ ገብቶ አንድ ሻይ ጠጣ። ሻይው በጣም ጣፋጭ ነበር፣ ነገር ግን አንድ ነገር 

tiny-aya-earth summary: median TTFT 0.17s, mean tok/s 15.7



[{'model': 'tiny-aya-earth',
  'prompt': 'ሰላም እንዴት ነህ',
  'ttft_sec': 0.205,
  'total_sec': 2.809,
  'tokens_generated': 42,
  'tokens_per_sec': 14.95,
  'response': 'ሰላም! እንዴት ነህ? እኔ Aya ነኝ፣ Cohere የፈጠረችው AI ረዳት። እንዴት ልረዳህ እችላለሁ?'},
 {'model': 'tiny-aya-earth',
  'prompt': 'የአዲስ አበባ የአየር ሁኔታ ዛሬ እንዴት ነው',
  'ttft_sec': 0.167,
  'total_sec': 7.606,
  'tokens_generated': 121,
  'tokens_per_sec': 15.91,
  'response': 'የአዲስ አበባ የአየር ሁኔታ ዛሬ (በጁን 2024) በአጠቃላይ **በጣም ሙቅ እና ደረቅ** ነው።\n\n- **ሙቀት:** ከ25°C እስከ 30°C (77°F እስከ 86°F) ይደርሳል።\n- **አየር ሁኔታ:** አብዛኛው ጊዜ **ፀሐይ ይበራል**፣ ግን አንዳንድ ጊዜ **አጭር ዝናብ** ሊያጋጥም ይችላል።\n- **አየር እርጥበት:** ዝቅተኛ (ከ30%'},
 {'model': 'tiny-aya-earth',
  'prompt': 'አንድ ቀላል የኢትዮጵያ ምግብ አዘገጃጀት ንገረኝ',
  'ttft_sec': 0.167,
  'total_sec': 7.671,
  'tokens_generated': 122,
  'tokens_per_sec': 15.9,
  'response': 'እርግጠኛ ነኝ! እዚህ ላይ አንድ ቀላል የኢትዮጵያ ምግብ አዘገጃጀት አለ፡ **ቲቦ (ቲቦ ስጋ)**።\n\n### **ቁሳቁሶች**\n- 500 ግራም የዶሮ ወይም የግመል ስጋ (በትንሹ የተቆረጠ)\n- 2 የሻይ ማንኪያ ዘይት\n- 1 የሻይ ማንኪያ ቀይ ሽንኩርት (በትንሽ የተቆረጠ)\n

In [7]:
# If this OOMs on a single T4, uncomment load_in_4bit below.
benchmark_llm(
    "google/gemma-4-E4B-it",
    prompts_am,
    model_label="gemma-4-E4B",
    # load_kwargs={"load_in_4bit": True},
)

Loading gemma-4-E4B ...


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Load time: 149.5s (one-time cost)
  2.36s | 18 tok -> ሰላም! ደህና ነኝ፣ አንተስ እንዴት ነህ? 😊
  14.52s | 120 tok -> እንደ ትልቅ የቋንቋ ሞዴል፣ የእውነተኛ ጊዜ የአየር ሁኔታ መረጃ የማግኘት አቅም የለኝም።

ዛሬ
  14.32s | 120 tok -> እርግጥ ነው! በጣም ቀላል እና ተወዳጅ የሆነ የኢትዮጵያ ምግብ የሆነውን **"ዶሮ ወጥ" (Dor
  7.82s | 65 tok -> እሺ፣ ነገ ስብሰባ እንዳለህ አስታውሳለሁ።

**ነገ ስብሰባ እንዳለህ እናስታውሳለሁ።**

ስብሰ
  2.11s | 17 tok -> የኢትዮጵያ ዋና ከተማ **አዲስ አበባ** ናት።
  14.35s | 120 tok -> እኔ የቋንቋ ሞዴል ነኝ፣ እና እኔ በቅጽበት ተመዝግበው ወይም እውነተኛ ጊዜ መረጃ የማግኘት አቅ
  14.31s | 120 tok -> እንደ ትልቅ የቋንቋ ሞዴል፣ እኔ የቅርብ ጊዜውን የጎዳና ትራፊክ ሁኔታ በቀጥታ ማየት አልችልም።
  14.35s | 119 tok -> እርግጠኛ ነኝ! እነሆ አንድ አጭር ቀልድ:

መምህር ለተማሪው፡ "የሕይወት ትምህርት ምንድነው?"

gemma-4-E4B summary: median TTFT 0.14s, mean tok/s 8.2



[{'model': 'gemma-4-E4B',
  'prompt': 'ሰላም እንዴት ነህ',
  'ttft_sec': 0.155,
  'total_sec': 2.359,
  'tokens_generated': 18,
  'tokens_per_sec': 7.63,
  'response': 'ሰላም! ደህና ነኝ፣ አንተስ እንዴት ነህ? 😊'},
 {'model': 'gemma-4-E4B',
  'prompt': 'የአዲስ አበባ የአየር ሁኔታ ዛሬ እንዴት ነው',
  'ttft_sec': 0.157,
  'total_sec': 14.515,
  'tokens_generated': 120,
  'tokens_per_sec': 8.27,
  'response': 'እንደ ትልቅ የቋንቋ ሞዴል፣ የእውነተኛ ጊዜ የአየር ሁኔታ መረጃ የማግኘት አቅም የለኝም።\n\nዛሬ በአዲስ አበባ የአየር ሁኔታ ምን እንደሚመስል ለማወቅ፣ እባክዎን የሚከተሉትን ያግኙ፦\n\n1.  **የአየር ሁኔታ ትንበያ መተግበሪያዎችን (Weather Apps) ይጠቀሙ:** (ለምሳሌ Google Weather, AccuWeather, The Weather Channel)\n2.  **የአየር ሁኔታ ድረ-ገጾችን (Websites) ይጎብኙ'},
 {'model': 'gemma-4-E4B',
  'prompt': 'አንድ ቀላል የኢትዮጵያ ምግብ አዘገጃጀት ንገረኝ',
  'ttft_sec': 0.141,
  'total_sec': 14.317,
  'tokens_generated': 120,
  'tokens_per_sec': 8.38,
  'response': 'እርግጥ ነው! በጣም ቀላል እና ተወዳጅ የሆነ የኢትዮጵያ ምግብ የሆነውን **"ዶሮ ወጥ" (Doro Wot)** አጭር እና ቀላል የሆነ አማራጭ ልሰጥህ እችላለሁ። ሙሉ ባህላዊውን የዶሮ ወጥ መስራት ትንሽ ረጅም ሂደት ስለሚሆን፣ ይሄን ግን በፍጥነት የሚያዘጋጅ መሠረታዊ

In [8]:
import pandas as pd
df = pd.read_csv("/kaggle/working/llm_benchmark_log.csv")
summary = df.groupby("model").agg(
    median_ttft=("ttft_sec", "median"),
    mean_tok_per_sec=("tokens_per_sec", "mean"),
).round(2)
print(summary)

                median_ttft  mean_tok_per_sec
model                                        
gemma-4-E4B            0.14              8.21
tiny-aya-earth         0.17             15.67
